<a href="https://colab.research.google.com/github/erickodiwuor014-max/plp-da-a1-gikomba-stall-analysis/blob/main/stall_analysis.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Gikomba Stall Analysis: How Is Wanjiku's Business Doing?

**Analyst:** Erick Odiwuor  |  **Cohort:** One Million Devs
**Assignment:** PLP Data Analytics, Week 1 (core Python only, no pandas or NumPy)

Wanjiku runs a fresh-produce stall at Gikomba Market. This notebook analyses her last 14 days of sales and her current stock to answer four questions: *How is the business doing? Is it growing? What should she restock? Where should she focus?*

## Dataset

The data below is copied exactly as provided in the assignment. Everything after this cell uses only these variables.

In [1]:
daily_sales = [2300, 1950, 2800, 2100, 3600, 5200, 4100,
               2500, 2200, 3100, 2400, 3900, 5600, 4700]   # 14 days, KES

days = ["Mon", "Tue", "Wed", "Thu", "Fri", "Sat", "Sun"] * 2

# qty = kg in stock, reorder = reorder level (kg), cost/price = KES per kg
stock = {
    "Tomatoes": {"qty": 18, "reorder": 25, "cost": 60, "price": 90},
    "Onions":   {"qty": 40, "reorder": 20, "cost": 70, "price": 100},
    "Sukuma":   {"qty": 9,  "reorder": 15, "cost": 30, "price": 50},
    "Potatoes": {"qty": 55, "reorder": 30, "cost": 50, "price": 75},
    "Avocado":  {"qty": 12, "reorder": 10, "cost": 80, "price": 140},
}

## Task 1: Total, average, highest and lowest daily sales

**Approach:** I use the built-in `sum()`, `len()`, `max()` and `min()` for the four headline numbers. To find *when* the best and worst days happened, I use `list.index()` to get the position of each value. Positions 0-6 are week 1 and 7-13 are week 2, so the week number is `position // 7 + 1` and the day name is `days[position]`.

In [2]:
total_sales = sum(daily_sales)
average_sales = total_sales / len(daily_sales)
highest_sales = max(daily_sales)
lowest_sales = min(daily_sales)

best_pos = daily_sales.index(highest_sales)
worst_pos = daily_sales.index(lowest_sales)

best_day, best_week = days[best_pos], best_pos // 7 + 1
worst_day, worst_week = days[worst_pos], worst_pos // 7 + 1

print(f"Total sales (14 days): KES {total_sales:,}")
print(f"Average daily sales:   KES {average_sales:,.2f}")
print(f"Highest day:           KES {highest_sales:,} ({best_day}, week {best_week})")
print(f"Lowest day:            KES {lowest_sales:,} ({worst_day}, week {worst_week})")


Total sales (14 days): KES 46,450
Average daily sales:   KES 3,317.86
Highest day:           KES 5,600 (Sat, week 2)
Lowest day:            KES 1,950 (Tue, week 1)


## Task 2: Week totals and week-on-week growth

**Approach:** `week_totals(sales)` slices the list into week 1 (`sales[:7]`) and week 2 (`sales[7:14]`), sums each slice and returns the two totals as a tuple. I unpack the tuple into two variables, then calculate growth as `(week 2 - week 1) / week 1 * 100`.

In [3]:
def week_totals(sales):
    """Return a tuple (week 1 total, week 2 total) for a list of 14 daily sales.

    Week 1 is the first 7 values (slice [:7]); week 2 is the next 7 (slice [7:14]).
    """
    return sum(sales[:7]), sum(sales[7:14])


week1_total, week2_total = week_totals(daily_sales)
growth_pct = (week2_total - week1_total) / week1_total * 100

print(f"Week 1 total: KES {week1_total:,}")
print(f"Week 2 total: KES {week2_total:,}")
print(f"Week-on-week growth: {growth_pct:+.2f}%")

Week 1 total: KES 22,050
Week 2 total: KES 24,400
Week-on-week growth: +10.66%


## Task 3: Classifying each day as Strong, Normal or Weak

**Approach:** `classify_day(amount, average)` compares a day's sales to the 14-day average. More than 20% above average (`> average * 1.2`) is **Strong**, more than 20% below (`< average * 0.8`) is **Weak**, and everything else is **Normal**. I apply it to every day in a loop and count the categories with the dictionary counting pattern (`dict.get(key, 0) + 1`).

In [4]:
def classify_day(amount, average):
    """Classify a day's sales against the average.

    Returns "Strong" if amount is more than 20% above average,
    "Weak" if more than 20% below average, otherwise "Normal".
    """
    if amount > average * 1.20:
        return "Strong"
    elif amount < average * 0.80:
        return "Weak"
    return "Normal"


day_labels = []                                   # (day name, amount, category) for each day
category_counts = {}                              # dictionary counting pattern

for day, amount in zip(days, daily_sales):
    category = classify_day(amount, average_sales)
    day_labels.append((day, amount, category))
    category_counts[category] = category_counts.get(category, 0) + 1

for position, (day, amount, category) in enumerate(day_labels, start=1):
    print(f"Day {position:>2} ({day}): KES {amount:>5,}  ->  {category}")

print()
print("Category counts:", category_counts)

Day  1 (Mon): KES 2,300  ->  Weak
Day  2 (Tue): KES 1,950  ->  Weak
Day  3 (Wed): KES 2,800  ->  Normal
Day  4 (Thu): KES 2,100  ->  Weak
Day  5 (Fri): KES 3,600  ->  Normal
Day  6 (Sat): KES 5,200  ->  Strong
Day  7 (Sun): KES 4,100  ->  Strong
Day  8 (Mon): KES 2,500  ->  Weak
Day  9 (Tue): KES 2,200  ->  Weak
Day 10 (Wed): KES 3,100  ->  Normal
Day 11 (Thu): KES 2,400  ->  Weak
Day 12 (Fri): KES 3,900  ->  Normal
Day 13 (Sat): KES 5,600  ->  Strong
Day 14 (Sun): KES 4,700  ->  Strong

Category counts: {'Weak': 6, 'Normal': 4, 'Strong': 4}


## Task 4: Weekdays (Mon-Fri) versus weekends (Sat-Sun)

**Approach:** I loop through the sales with their day names. Sat and Sun go into a `weekend_sales` list and the other days into `weekday_sales`. Each average is `sum / len`, and the difference is shown as a percentage of the weekday average.

In [5]:
weekday_sales = []
weekend_sales = []

for day, amount in zip(days, daily_sales):
    if day in ("Sat", "Sun"):
        weekend_sales.append(amount)
    else:
        weekday_sales.append(amount)

weekday_avg = sum(weekday_sales) / len(weekday_sales)
weekend_avg = sum(weekend_sales) / len(weekend_sales)
weekend_uplift_pct = (weekend_avg - weekday_avg) / weekday_avg * 100

print(f"Weekday average (Mon-Fri, {len(weekday_sales)} days): KES {weekday_avg:,.2f}")
print(f"Weekend average (Sat-Sun, {len(weekend_sales)} days): KES {weekend_avg:,.2f}")
print(f"Weekend sales are {weekend_uplift_pct:.2f}% higher than weekday sales")

Weekday average (Mon-Fri, 10 days): KES 2,685.00
Weekend average (Sat-Sun, 4 days): KES 4,900.00
Weekend sales are 82.50% higher than weekday sales


## Task 5: Reorder list and gross margin per product

**Approach:** `get_reorder_list(stock)` loops through `stock.items()` and collects every product whose `qty` is **at or below** its `reorder` level. A second function, `gross_margin_pct()`, applies `(price - cost) / price * 100`, and I loop with `.items()` to calculate it for every product.

In [6]:
def get_reorder_list(stock):
    """Return a list of product names whose quantity is at or below the reorder level."""
    to_reorder = []
    for product, info in stock.items():
        if info["qty"] <= info["reorder"]:
            to_reorder.append(product)
    return to_reorder


def gross_margin_pct(price, cost):
    """Return gross margin as a percentage: (price - cost) / price * 100."""
    return (price - cost) / price * 100


reorder_list = get_reorder_list(stock)
print("Products to reorder:", reorder_list)
print()

margins = {}
for product, info in stock.items():
    margins[product] = gross_margin_pct(info["price"], info["cost"])
    shortfall = max(info["reorder"] - info["qty"], 0)
    print(f"{product:<10} margin: {margins[product]:5.1f}%   "
          f"stock {info['qty']:>2} kg vs reorder level {info['reorder']:>2} kg   "
          f"(shortfall {shortfall} kg)")

Products to reorder: ['Tomatoes', 'Sukuma']

Tomatoes   margin:  33.3%   stock 18 kg vs reorder level 25 kg   (shortfall 7 kg)
Onions     margin:  30.0%   stock 40 kg vs reorder level 20 kg   (shortfall 0 kg)
Sukuma     margin:  40.0%   stock  9 kg vs reorder level 15 kg   (shortfall 6 kg)
Potatoes   margin:  33.3%   stock 55 kg vs reorder level 30 kg   (shortfall 0 kg)
Avocado    margin:  42.9%   stock 12 kg vs reorder level 10 kg   (shortfall 0 kg)


## Task 6: Stall Performance Report

**Approach:** I pull together the results from Tasks 1-5 and print them with f-strings. Money uses `KES {value:,}` for thousands separators, and fixed-width format specifiers (`<`, `>`, `^`) keep the columns aligned.

In [7]:
WIDTH = 62
LINE = "=" * WIDTH
THIN = "-" * WIDTH

print(LINE)
print("STALL PERFORMANCE REPORT".center(WIDTH))
print("Wanjiku's Fresh Produce Stall, Gikomba Market".center(WIDTH))
print("Period: 14 days (2 weeks)".center(WIDTH))
print(LINE)

print("\nSALES SUMMARY")
print(THIN)
print(f"{'Total sales':<26}KES {total_sales:>12,}")
print(f"{'Average daily sales':<26}KES {average_sales:>12,.2f}")
print(f"{'Best day':<26}KES {highest_sales:>12,}   ({best_day}, week {best_week})")
print(f"{'Worst day':<26}KES {lowest_sales:>12,}   ({worst_day}, week {worst_week})")

print("\nGROWTH")
print(THIN)
print(f"{'Week 1 total':<26}KES {week1_total:>12,}")
print(f"{'Week 2 total':<26}KES {week2_total:>12,}")
print(f"{'Week-on-week growth':<26}{growth_pct:>+16.2f}%")

print("\nDAY PERFORMANCE (vs. average)")
print(THIN)
for category in ("Strong", "Normal", "Weak"):
    print(f"{category + ' days':<26}{category_counts.get(category, 0):>16}")

print("\nWEEKDAYS vs WEEKENDS")
print(THIN)
print(f"{'Weekday average (Mon-Fri)':<26}KES {weekday_avg:>12,.2f}")
print(f"{'Weekend average (Sat-Sun)':<26}KES {weekend_avg:>12,.2f}")
print(f"{'Weekend uplift':<26}{weekend_uplift_pct:>+16.2f}%")

print("\nSTOCK STATUS")
print(THIN)
print(f"{'Product':<12}{'In stock':>9}{'Reorder':>9}{'Margin':>9}   Status")
for product, info in stock.items():
    status = "REORDER" if product in reorder_list else "OK"
    print(f"{product:<12}{info['qty']:>6} kg{info['reorder']:>6} kg{margins[product]:>8.1f}%   {status}")

print("\n" + LINE)

                   STALL PERFORMANCE REPORT                   
        Wanjiku's Fresh Produce Stall, Gikomba Market         
                  Period: 14 days (2 weeks)                   

SALES SUMMARY
--------------------------------------------------------------
Total sales               KES       46,450
Average daily sales       KES     3,317.86
Best day                  KES        5,600   (Sat, week 2)
Worst day                 KES        1,950   (Tue, week 1)

GROWTH
--------------------------------------------------------------
Week 1 total              KES       22,050
Week 2 total              KES       24,400
Week-on-week growth                 +10.66%

DAY PERFORMANCE (vs. average)
--------------------------------------------------------------
Strong days                              4
Normal days                              4
Weak days                                6

WEEKDAYS vs WEEKENDS
--------------------------------------------------------------
Weekday average (Mon

## Task 7: Business recommendations

**Approach:** Each recommendation below is tied to a specific number from Tasks 1-6, so Wanjiku can see exactly why it is being made. The recommendations are in the final section.

## Recommendations for Wanjiku

1. **Restock Tomatoes and Sukuma before the weekend.** Tomatoes (18 kg vs a 25 kg reorder level) and Sukuma (9 kg vs 15 kg) are the only products below their reorder level, and weekend sales average KES 4,900 a day, 82.5% more than the KES 2,685 weekday average, so running out on a Saturday would cost her the most.
2. **Focus on weekends and buy lighter early in the week.** All 4 Strong days were Saturdays or Sundays, while all 6 Weak days fell on Mon, Tue or Thu (as low as KES 1,950), so she should buy smaller stock and run a small offer on those days.
3. **Push Avocado and Sukuma, and do not reorder Potatoes.** Avocado earns the best margin (42.9%) and Sukuma the next best (40.0%), while Potatoes (33.3%) already have 55 kg in stock, far above the 30 kg reorder level.
4. **Keep doing what is working.** Sales grew 10.66% from KES 22,050 to KES 24,400, and she should keep recording daily sales so she can check whether the growth continues.